# Simplified Pipeline — Stock Price Feature Engineering
Versi ringkas dari `Main_Notebook.ipynb`. Semua sel diagnostik/EDA panjang dihapus, penanganan missing value & outlier dibuat langsung tanpa analisis bertele-tele.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

## 2. Load Dataset

In [ ]:
df = pd.read_csv("stock_prices.csv")

keep_cols = ["RowId", "Date", "SecuritiesCode", "Open", "High", "Low", "Close", "Volume", "Target"]
df = df[keep_cols].copy()

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(f"Date range: {df['Date'].min().date()} -> {df['Date'].max().date()}")

## 3. Split Train/Test (berbasis waktu, 80/20)
Split berdasarkan tanggal (bukan random) supaya tidak ada kebocoran data masa depan ke train.

In [ ]:
split_date = df["Date"].quantile(0.8)

train = df[df["Date"] <= split_date].copy()
test  = df[df["Date"] >  split_date].copy()

print(f"Train: {train['Date'].min().date()} -> {train['Date'].max().date()}  ({len(train)} rows)")
print(f"Test : {test['Date'].min().date()} -> {test['Date'].max().date()}  ({len(test)} rows)")

## 4. Missing Values — Drop Langsung
Tidak ada analisis missing value per kolom/tahun/saham seperti notebook asli. Baris yang punya NaN di kolom manapun langsung dibuang dengan `dropna()`.

In [ ]:
print(f"Sebelum drop NaN -> train: {train.shape}, test: {test.shape}")

train = train.dropna().reset_index(drop=True)
test  = test.dropna().reset_index(drop=True)

print(f"Setelah drop NaN -> train: {train.shape}, test: {test.shape}")
print(f"Sisa missing value -> train: {train.isna().sum().sum()}, test: {test.isna().sum().sum()}")

## 5. Fitur Temporal
Ambil komponen tanggal (year, month, dayofweek) sebagai fitur numerik.

In [ ]:
for d in (train, test):
    d["year"]      = d["Date"].dt.year
    d["month"]     = d["Date"].dt.month
    d["dayofweek"] = d["Date"].dt.dayofweek

print("Fitur temporal ditambahkan: year, month, dayofweek")

## 6. Encoding SecuritiesCode — Mean Encoding Sederhana
Versi asli pakai target encoding out-of-fold (K-Fold) + smoothing supaya tidak leak. Di sini disederhanakan jadi mean encoding biasa: dihitung dari `train`, lalu di-*map* ke `test` (kode saham yang tidak muncul di train diisi rata-rata global).

In [ ]:
mean_target = train.groupby("SecuritiesCode")["Target"].mean()
global_mean = train["Target"].mean()

train["SecuritiesCode_te"] = train["SecuritiesCode"].map(mean_target)
test["SecuritiesCode_te"]  = test["SecuritiesCode"].map(mean_target).fillna(global_mean)

print(f"SecuritiesCode_te dibuat. mean={train['SecuritiesCode_te'].mean():.6f}")

## 7. Fitur Return, Volatilitas & Volume
Fitur turunan dari harga & volume dihitung per saham, memakai data historis saja.

In [ ]:
train["_split"] = "train"
test["_split"]  = "test"
full = pd.concat([train, test], axis=0, ignore_index=True)
full = full.sort_values(["SecuritiesCode", "Date"]).reset_index(drop=True)

g = full.groupby("SecuritiesCode", sort=False)

full["ret_1d"]          = g["Close"].pct_change()
full["intraday_range"]  = (full["High"] - full["Low"]) / full["Close"]
full["close_open"]      = (full["Close"] - full["Open"]) / full["Open"]
full["log_volume"]      = np.log1p(full["Volume"])
full["vol_5d"]          = g["ret_1d"].transform(lambda s: s.rolling(5, min_periods=2).std())

print(f"Fitur return/volatilitas ditambahkan. Shape: {full.shape}")

## 8. Outlier Handling — Capping IQR Langsung
Versi asli punya fungsi `handle_outliers()` dengan pilihan method (IQR/Z-score) dan action (cap/remove) plus visualisasi boxplot. Di sini disederhanakan jadi satu fungsi capping IQR yang langsung diterapkan ke fitur return.

In [ ]:
def cap_outliers_iqr(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series.clip(lower, upper)

full["ret_1d"] = cap_outliers_iqr(full["ret_1d"])

print("Outlier pada 'ret_1d' sudah di-cap dengan batas IQR (1.5x).")

## 9. Finalisasi Fitur
Buang kolom OHLCV mentah, buang NaN sisa dari rolling/pct_change (langsung `dropna()`), pisahkan kembali train/test, lalu buang kolom identifier yang tidak dipakai sebagai fitur.

In [ ]:
full = full.drop(columns=["Open", "High", "Low", "Close", "Volume"])
full = full.dropna().reset_index(drop=True)

train_final = full[full["_split"] == "train"].drop(columns=["_split", "RowId", "SecuritiesCode", "Date"]).reset_index(drop=True)
test_final  = full[full["_split"] == "test"].drop(columns=["_split", "RowId", "SecuritiesCode", "Date"]).reset_index(drop=True)

print(f"Fitur final ({len(train_final.columns)} kolom): {list(train_final.columns)}")
print(f"train_final: {train_final.shape} | test_final: {test_final.shape}")

## 10. Feature Scaling
Pisahkan X/y, lalu scaling untuk dua pipeline model: StandardScaler (SVR) dan MinMaxScaler (LSTM). Scaler hanya di-*fit* pada train, supaya tidak ada kebocoran statistik dari test.

In [ ]:
FEATURE_COLS = [c for c in train_final.columns if c != "Target"]
TARGET_COL = "Target"

X_train = train_final[FEATURE_COLS].values.astype(np.float64)
y_train = train_final[TARGET_COL].values.astype(np.float64).reshape(-1, 1)
X_test  = test_final[FEATURE_COLS].values.astype(np.float64)
y_test  = test_final[TARGET_COL].values.astype(np.float64).reshape(-1, 1)

# --- Pipeline SVR ---
scaler_X_svr = StandardScaler()
scaler_y_svr = StandardScaler()
X_train_svr = scaler_X_svr.fit_transform(X_train)
X_test_svr  = scaler_X_svr.transform(X_test)
y_train_svr = scaler_y_svr.fit_transform(y_train).ravel()
y_test_svr  = scaler_y_svr.transform(y_test).ravel()

# --- Pipeline LSTM ---
scaler_X_lstm = MinMaxScaler()
scaler_y_lstm = MinMaxScaler()
X_train_lstm = scaler_X_lstm.fit_transform(X_train)
X_test_lstm  = scaler_X_lstm.transform(X_test)
y_train_lstm = scaler_y_lstm.fit_transform(y_train).ravel()
y_test_lstm  = scaler_y_lstm.transform(y_test).ravel()

print(f"Jumlah fitur: {len(FEATURE_COLS)}")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")